### Inputing

In [ ]:
import pandas as pd
import re
import numpy as np

In [ ]:
df = pd.read_csv("f1nalyze-datathon-ieeecsmuj\\train.csv",low_memory=False)
test_df = pd.read_csv("f1nalyze-datathon-ieeecsmuj\\test.csv",low_memory=False)
val_df = pd.read_csv("f1nalyze-datathon-ieeecsmuj\\validation.csv",low_memory=False)
# TODO Assign dtype on entry

### EDA

In [ ]:
for dataframe in [df,test_df,val_df]:
    dataframe = dataframe.set_index("result_driver_standing")

In [ ]:
df.head()

In [ ]:
headers = df.columns.tolist()
print(headers)

In [ ]:
# df["fastestLapProcessed"].unique()

### Attribute Dropping and Data Types

In [ ]:
null_cols = ['quali_date', 'sprint_time', 'sprint_date' ,'quali_time']

df = df.drop(columns=null_cols)
test_df = test_df.drop(columns=null_cols)
val_df = val_df.drop(columns=null_cols)
df = df.drop(columns = ['positionText_y',])
val_df = val_df.drop(columns = ['positionText_y',])

In [ ]:
less_useful_cols = ['fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time','url','url_y', 'url_x']

df = df.drop(columns=less_useful_cols)
test_df = test_df.drop(columns=less_useful_cols)
val_df = val_df.drop(columns=less_useful_cols)

In [ ]:
float_attribs = ['fastestLap','max_speed','wins', 'points_y', 'racerId', 'driverId', 'constructorId', 'circuitId', 'statusId', 'grid', 'laps', 'timetaken_in_millisec', 'points', 'year', 'positionOrder', 'round','resultId']

for attrib in float_attribs:
    for dataframe in [df,test_df,val_df]:
        dataframe[attrib] = pd.to_numeric(dataframe[attrib], errors='coerce')

In [ ]:
for dataframe in [test_df,df,val_df]:
    dataframe["dob_yr"] = dataframe["dob"].apply(lambda x: x.split("-")[0])
    dataframe["dob_yr"] = pd.to_numeric(dataframe["dob_yr"], errors='coerce')

    dataframe["date_yr"] = dataframe["date"].apply(lambda x: x.split("-")[0])
    dataframe["date_yr"] = pd.to_numeric(dataframe["date_yr"], errors='coerce')

    dataframe["date_month"] = dataframe["date"].apply(lambda x: x.split("-")[1]
                                                      if len(x.split("-"))>1
                                                      else np.nan)
    dataframe["date_month"] = pd.to_numeric(dataframe["date_month"], errors='coerce')

    dataframe["time_y_processed"] = dataframe["time_y"].apply(lambda x: float(x.split(":")[0])+float(x.split(":")[1])/60
                                                              if len(x.split(":"))>1
                                                              else np.nan)
    dataframe["fastestLapProcessed"] = dataframe["fastestLapTime"].apply(lambda x: float(x.split(":")[0])+float((x.split(":")[1])[0:2])/60+float((x.split(".")[1]))/3600
                                                                        if len(x.split(":"))>1
                                                                        else np.nan)

In [ ]:
post_proc_cols = ["dob","date","time_y","fastestLapTime"]

df = df.drop(columns=post_proc_cols)
test_df = test_df.drop(columns=post_proc_cols)
val_df = val_df.drop(columns=post_proc_cols)

In [ ]:
# dataframe["date_yr"].unique()

In [ ]:
dupe_cols = ['number','rank','driver_code', 'forename', 'surname', 'driverRef', 'driver_num','company','time_x','status','position_x', 'positionText_x','constructorRef']

df = df.drop(columns=dupe_cols)
test_df = test_df.drop(columns=dupe_cols)
val_df = val_df.drop(columns=dupe_cols)

In [ ]:
temp_cols = ['driverStandingsId' ,'raceId_y']

df = df.drop(columns=temp_cols)
test_df = test_df.drop(columns=temp_cols)
val_df = val_df.drop(columns=temp_cols)

In [ ]:
df.head()

In [ ]:
headers = df.columns.tolist()
print(headers)

In [ ]:
from sklearn.preprocessing import LabelEncoder

for attribute in ['nationality','nationality_y','grand_prix']:#
    label_encoder = LabelEncoder()
    label_encoder.fit(pd.concat([df[attribute],test_df[attribute],val_df[attribute]],ignore_index=True))
    df[attribute] = label_encoder.transform(df[attribute])
    test_df[attribute] = label_encoder.transform(test_df[attribute])
    val_df[attribute] = label_encoder.transform(val_df[attribute])

In [ ]:
df

### Split

In [ ]:
X_train = df.drop(columns=["position","result_driver_standing"])
Y_train = df["position"]
X_val = val_df.drop(columns=["position"])
Y_val = val_df["position"]
X_test = test_df

### Models

In [ ]:
# from sklearn.metrics import mean_squared_error
# import numpy as np

# from sklearn.neural_network import MLPRegressor
# from sklearn.ensemble import RandomForestRegressor

In [ ]:
# mdl = MLPRegressor(activation ='relu', solver='adam',max_iter=2000)
# mdl.fit(X_train,Y_train)
# pdc = mdl.predict(X_val)
# rmse = np.sqrt(mean_squared_error(Y_val, pdc))
# print("score: {0:,.0f}".format(rmse))

### LGBM

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
d_train = lgb.Dataset(X_train, label=Y_train)

params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.001,
}

mdl = lgb.train(params, d_train, num_boost_round=10000)

# Make predictions
pdc = mdl.predict(X_val.drop(columns=["result_driver_standing"]))
rmse = np.sqrt(mean_squared_error(Y_val, pdc))
print("score: {0:,.0f}".format(rmse))

In [ ]:
import matplotlib.pyplot as plt
lgb.plot_importance(mdl)
plt.show()

In [ ]:
# import lightgbm as lgb
# import numpy as np
# from sklearn.metrics import mean_squared_error
# from sklearn.model_selection import GridSearchCV

In [ ]:
# d_train = lgb.Dataset(X_train, label=Y_train)

# params = {
#     'boosting_type': 'gbdt',
#     'metric': 'mse'
# }

# param_grid = {
#     'learning_rate': [0.01, 0.05],
#     'n_estimators': [500, 1000],
#     'reg_alpha': [0, 0.1, 0.5],
#     'reg_lambda': [0, 0.1, 0.5],
# }

# lgbm = lgb.LGBMRegressor(**params)
# grid_search = GridSearchCV(estimator=lgbm, param_grid=param_grid, cv=5, scoring='r2', verbose=1)
# grid_search.fit(X_train, Y_train)

In [ ]:
# best_params = grid_search.best_params_
# print(f"Best parameters found: {best_params}")

In [ ]:
# params = {
#     'boosting_type': 'gbdt',
#     'metric': 'mse'
# }

# best_params = {'learning_rate': 0.01, 'n_estimators': 500, 'reg_alpha': 0.5, 'reg_lambda': 0.1}

# mdl = lgb.LGBMRegressor(**params, **best_params)
# mdl.fit(X_train, Y_train)

In [ ]:
# pdc = mdl.predict(X_val.drop(columns=["result_driver_standing"]))
# rmse = np.sqrt(mean_squared_error(Y_val, pdc))
# print("score: {0:,.0f}".format(rmse))

### Outputting

In [ ]:
import csv

predictions = mdl.predict(X_test.drop(columns=["result_driver_standing"]))

filename = "test_output_4.csv"
with open(filename, mode='w', newline="") as csv_file:
    fieldnames = ['position', 'result_driver_standing']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    writer.writeheader()
    for i in range(len(predictions)):
        writer.writerow({'position': predictions[i], 'result_driver_standing': X_test["result_driver_standing"].iloc[i]})